# 🏥 Notebook 1 — Foundations: From Bedrock to Live Agent

In this notebook you'll:
1. Make your first Bedrock API call (zero config — IAM role handles auth)
2. Connect to a **live Teradata HCLS database** via MCP
3. Build an agent with **Strands SDK** that queries real healthcare data
4. Ask questions in natural language and watch SQL execute in real-time

**No API keys. No `.env` files. Just run the cells.**

---
## 1.1 — Install Dependencies

In [ ]:
!pip install --quiet strands-agents mcp boto3 python-dotenv 2>/dev/null

# Install uv (needed for Teradata MCP Server)
!curl -LsSf https://astral.sh/uv/install.sh | sh 2>/dev/null
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:{os.environ['PATH']}"
!uv --version

print("All dependencies installed.")

---
## 1.2 — Your First Bedrock Call

The SageMaker notebook has an IAM role. `boto3` just works — no credentials needed.

In [ ]:
import boto3
import json

bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

response = bedrock.invoke_model(
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    body=json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 512,
        "messages": [{"role": "user", "content": "What is a prior authorization in healthcare? Keep it to 2 sentences."}]
    })
)

result = json.loads(response["body"].read())
print(result["content"][0]["text"])

That's raw Bedrock — a model you can talk to, but it can't *do* anything.  
Now let's give it access to a real database.

---
## 1.3 — Connect to Teradata via MCP

The **Teradata MCP Server** is a bridge between the AI and the database.  
It translates tool calls into SQL and returns results.

In [ ]:
!pip install --quiet python-dotenv 2>/dev/null
from dotenv import load_dotenv
load_dotenv()

from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

# Teradata MCP Server — connects to the HCLS database
server_params = StdioServerParameters(
    command="uvx",
    args=["teradata-mcp-server"],
    env={
        "DATABASE_URI": os.getenv("TERADATA_DATABASE_URI"),
        "LOGMECH": os.getenv("LOGMECH", "TD2")
    }
)
teradata_tool = MCPClient(
    lambda: stdio_client(server_params),
    startup_timeout=300
)

print("Teradata MCP client configured.")
print("Database: HCLS (Healthcare Claims & Prior Auth)")
uri = os.getenv("TERADATA_DATABASE_URI", "")
host = uri.split("@")[1] if "@" in uri else "(check .env)"
print(f"Connection: {host}")


---
## 1.4 — Create a Strands Agent

A **Strands Agent** combines:
- A model (Claude on Bedrock)
- Tools (Teradata MCP)
- A system prompt (who the agent is)

The agent loop — think → call tool → observe → repeat — is handled by the framework.

In [ ]:
from strands import Agent
from strands.models.bedrock import BedrockModel

# Model — Claude on Bedrock, IAM role handles auth
model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    streaming=False
)

# System prompt — this is what makes it "Claire"
SYSTEM_PROMPT = """
You are Claire, an intelligent healthcare claims and prior authorization AI agent.
You have access to a Teradata HCLS database via MCP tools.

Key tables in the HCLS database:
- prior_auth_request: PA headers (auth_request_id, auth_request_num, member_id, ordering_provider_id, facility_id, auth_status_id)
- member: Members (member_id, member_id_num like 'MBR-884201', first_name, last_name, date_of_birth, gender)
- provider: Providers (provider_id, npi, first_name, last_name, primary_specialty, credential)
- facility: Facilities (facility_id, facility_name, facility_type, city, state_code)
- auth_cpt_code: CPT codes on a PA (auth_request_id, cpt_code)
- cpt_codes: CPT descriptions (cpt_code, cpt_desc)
- auth_icd_code: ICD diagnosis codes on a PA
- auth_clinical_notes: Clinical notes and prior test results
- auth_status: Status lookup (auth_status_id, auth_status_code, auth_status_name)
- care_pathways: Prerequisite rules (cpt_code, cpt_code_prereq)
- WORKERS_COMP_HDR / WORKERS_COMP_DTL: Claims history
- alerts: Fraud/integrity alerts (object_id = provider NPI)

CRITICAL rules:
- Provider FK is ordering_provider_id, NOT provider_id
- Specialty column is primary_specialty, NOT specialty
- Use Teradata SQL (TOP instead of LIMIT)
- alerts table has NO npi column — use object_id

Focus on answering the question asked with data.
"""

print("Model and system prompt configured.")
print("Ready to create the agent in the next cell.")

---
## 1.5 — Run the Agent Against Live Data

This is the moment — the agent will connect to Teradata, discover tables,  
write SQL, execute it, and return results. All from a natural language question.

In [ ]:
# Create the agent — model + tools + prompt
agent = Agent(
    model=model,
    tools=[teradata_tool],
    system_prompt=SYSTEM_PROMPT
)

# Ask it something!
result = agent("What tables are available in the HCLS database? Show me a brief summary.")
print(result.message)

In [ ]:
# Query real PA data
result = agent("Show me the 5 most recent prior authorization requests with member names and status.")
print(result.message)

In [ ]:
# Look up a specific PA
result = agent("Look up PA-2026-003417. Show me the member, provider, facility, and CPT codes.")
print(result.message)

In [ ]:
# Check for fraud alerts
result = agent("Check if there are any fraud alerts for the provider on PA-2026-003417. Get the provider NPI first, then check the alerts table using object_id.")
print(result.message)

---
## 1.6 — What Just Happened?

In ~20 lines of Python you built an **AI agent** that:

1. **Connects to Bedrock** — Claude provides the reasoning (IAM role = zero config)
2. **Connects to Teradata** — MCP server translates tool calls to SQL
3. **Runs an agent loop** — Strands SDK handles think → act → observe → repeat
4. **Queries live data** — Real PA requests, real members, real providers

The same pattern powers the full Claire production app.  
In the next notebook, we'll add **skills** — modular instructions that tell  
the agent *how* to handle different types of requests.

---

**➡ Continue to [Notebook 2 — Skills & Claire Deep Dive](02-claire-deep-dive.ipynb)**